Identical split to training: seed 42, 90/10

In [ ]:
import json, random, math, os, gc
import numpy as np, pandas as pd, torch
 
DATASET_PATH = "tanglish.jsonl"
ADAPTER_DIR  = "tanglish_gemma2b_adapter"
BASE_ID      = "google/gemma-2b"
IT_ID        = "google/gemma-2b-it"
MAX_NEW      = 200
RESULTS_DIR  = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
 
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw = [json.loads(l) for l in f]
 
def fmt(r):
    ins, inp, out = (r.get("instruction", "").strip(), r.get("input", "").strip(), r.get("output", "").strip())
    user = f"{ins}\n\nContext:\n{inp}" if inp else ins
    return {
        "text": f"<start_of_turn>user\n{user}<end_of_turn>\n<start_of_turn>model\n{out}<end_of_turn>",
        "instruction": ins, "input": inp, "output": out,
    }
 
formatted = [fmt(r) for r in raw]
random.seed(42)
random.shuffle(formatted)
cut = int(len(formatted) * 0.9)
train, val = formatted[:cut], formatted[cut:]
print(f"Train {len(train)} | Val {len(val)}")
 
MODEL_TAG = "<start_of_turn>model\n"
def split_prompt_target(text):
    prompt = text.split(MODEL_TAG)[0] + MODEL_TAG
    target = text.split(MODEL_TAG)[-1].replace("<end_of_turn>", "").strip()
    return prompt, target
 
val_prompts, val_refs = zip(*[split_prompt_target(v["text"]) for v in val])
val_prompts, val_refs = list(val_prompts), list(val_refs)


Fuzzy leakage check (near-duplicates, not just exact)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
 
def near_dup_report(train_texts, val_texts, name, thr=0.90):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(train_texts + val_texts)
    sims = cosine_similarity(vec.transform(val_texts), vec.transform(train_texts))
    best = sims.max(axis=1)
    print(f"[{name}] mean best-sim {best.mean():.3f} | max {best.max():.3f} | "
          f"val items with sim >= {thr}: {(best >= thr).sum()}/{len(best)}")
    return best
 
best_prompt = near_dup_report([t["instruction"] + " " + t["input"] for t in train],
                              [v["instruction"] + " " + v["input"] for v in val], "prompts")
best_output = near_dup_report([t["output"] for t in train],
                              [v["output"] for v in val], "outputs")
pd.DataFrame({"prompt_sim": best_prompt, "output_sim": best_output}).to_csv(
    f"{RESULTS_DIR}/leakage_fuzzy.csv", index=False)


Load models: base+LoRA (one object, adapter toggled) and Gemma-2B-it

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
 
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE_ID)
tok.pad_token = tok.eos_token
END_ID = tok.convert_tokens_to_ids("<end_of_turn>")
 
base = AutoModelForCausalLM.from_pretrained(BASE_ID, quantization_config=bnb, device_map="auto")
ft_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
ft_model.eval()
 
it_model = AutoModelForCausalLM.from_pretrained(IT_ID, quantization_config=bnb, device_map="auto")
it_model.eval()
 


Core functions: response-only NLL (token-weighted) and generation

In [ ]:
import torch.nn.functional as F
 
@torch.no_grad()
def response_nll(model, text, prompt):
    """Return (sum NLL over response tokens, number of response tokens)."""
    ids = tok(text, return_tensors="pt").input_ids.to("cuda")
    p_len = len(tok(prompt).input_ids)
    logits = model(input_ids=ids).logits[:, :-1, :].float()
    targets = ids[:, 1:]
    nll = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), reduction="none")
    nll = nll.reshape(targets.shape)[:, p_len - 1:]        # response tokens only
    return nll.sum().item(), nll.numel()
 
@torch.no_grad()
def generate(model, prompt):
    inp = tok(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inp, max_new_tokens=MAX_NEW, do_sample=False,
                         eos_token_id=[tok.eos_token_id, END_ID], pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
 
def run_model(name, model, use_adapter=None):
    """use_adapter: None (plain model), True (adapter on), False (adapter off)."""
    nll_sum, ntok, preds = [], [], []
    for v, prompt in zip(val, val_prompts):
        if use_adapter is False:
            with model.disable_adapter():
                s, n = response_nll(model, v["text"], prompt)
                p = generate(model, prompt)
        else:
            s, n = response_nll(model, v["text"], prompt)
            p = generate(model, prompt)
        nll_sum.append(s); ntok.append(n); preds.append(p)
    print(f"{name}: done")
    return {"nll_sum": np.array(nll_sum), "ntok": np.array(ntok), "preds": preds}

Run all three systems on the validation set

In [ ]:
runs = {
    "base":        run_model("base", ft_model, use_adapter=False),
    "finetuned":   run_model("finetuned", ft_model, use_adapter=True),
    "gemma2b_it":  run_model("gemma2b_it", it_model, use_adapter=None),
}

Per-sample metrics: ROUGE, chrF, BERTScore

In [ ]:
import evaluate
from sacrebleu.metrics import CHRF
 
rouge = evaluate.load("rouge")
chrf = CHRF()
 
def per_sample_metrics(preds, refs):
    r = rouge.compute(predictions=preds, references=refs, use_aggregator=False)
    out = {"rouge1": np.array(r["rouge1"]), "rouge2": np.array(r["rouge2"]), "rougeL": np.array(r["rougeL"]),
           "chrf": np.array([chrf.sentence_score(p, [t]).score for p, t in zip(preds, refs)])}
    try:
        from bert_score import score as bscore
        _, _, f1 = bscore(preds, refs, model_type="bert-base-multilingual-cased", verbose=False)
        out["bertscore_f1"] = f1.numpy()
    except Exception as e:
        print("BERTScore skipped:", e)
    return out
 
for k, r in runs.items():
    r["metrics"] = per_sample_metrics(r["preds"], val_refs)


Bootstrap CIs + paired significance

In [ ]:
rng = np.random.default_rng(0)
B = 10000
N = len(val)
idx_boot = rng.integers(0, N, size=(B, N))   # same resamples for every system -> paired
 
def ppl_from(sums, toks, idx):
    return np.exp(sums[idx].sum(axis=1) / toks[idx].sum(axis=1))
 
def ci(arr):
    return float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))
 
rows = []
for name, r in runs.items():
    full = math.exp(r["nll_sum"].sum() / r["ntok"].sum())
    lo, hi = ci(ppl_from(r["nll_sum"], r["ntok"], idx_boot))
    rows.append({"system": name, "metric": "response_PPL (lower=better)", "value": full, "ci_low": lo, "ci_high": hi})
    for m, vals in r["metrics"].items():
        lo, hi = ci(vals[idx_boot].mean(axis=1))
        rows.append({"system": name, "metric": m, "value": float(vals.mean()), "ci_low": lo, "ci_high": hi})
 
summary = pd.DataFrame(rows)
summary.to_csv(f"{RESULTS_DIR}/rq1_summary_with_ci.csv", index=False)
print(summary.round(4).to_string(index=False))
 
# Paired bootstrap: finetuned vs each baseline
def paired(a, b, label):
    out = []
    pa = ppl_from(runs[a]["nll_sum"], runs[a]["ntok"], idx_boot)
    pb = ppl_from(runs[b]["nll_sum"], runs[b]["ntok"], idx_boot)
    d = pb - pa    # PPL: positive = a (finetuned) better
    out.append({"comparison": label, "metric": "PPL (b - a)", "mean_diff": d.mean(), "ci_low": ci(d)[0],
                "ci_high": ci(d)[1], "p_a_not_better": float((d <= 0).mean())})
    for m in runs[a]["metrics"]:
        d = (runs[a]["metrics"][m][idx_boot].mean(axis=1) - runs[b]["metrics"][m][idx_boot].mean(axis=1))
        out.append({"comparison": label, "metric": m + " (a - b)", "mean_diff": d.mean(), "ci_low": ci(d)[0],
                    "ci_high": ci(d)[1], "p_a_not_better": float((d <= 0).mean())})
    return out
 
paired_df = pd.DataFrame(paired("finetuned", "base", "finetuned vs base") +
                         paired("finetuned", "gemma2b_it", "finetuned vs gemma2b_it"))
paired_df.to_csv(f"{RESULTS_DIR}/rq1_paired_tests.csv", index=False)
print("\n", paired_df.round(4).to_string(index=False))
# Reading it: if the CI of the difference excludes 0, the gain is statistically supported.
 
# Optional: re-run on low-similarity subset to show leakage does not drive results
mask = (best_prompt < 0.90) & (best_output < 0.90)
print(f"\nVal items kept after near-duplicate filter: {mask.sum()}/{N}")
for k, r in runs.items():
    print(k, "PPL (filtered):", round(math.exp(r["nll_sum"][mask].sum() / r["ntok"][mask].sum()), 2),
          "| ROUGE-L (filtered):", round(r["metrics"]["rougeL"][mask].mean(), 4))


Save predictions for manual reading (20 random samples)

In [ ]:
pred_df = pd.DataFrame({"prompt": val_prompts, "reference": val_refs,
                        "base": runs["base"]["preds"], "finetuned": runs["finetuned"]["preds"],
                        "gemma2b_it": runs["gemma2b_it"]["preds"]})
pred_df.to_csv(f"{RESULTS_DIR}/val_predictions_all_systems.csv", index=False)
pred_df.sample(20, random_state=1).to_csv(f"{RESULTS_DIR}/manual_review_sample.csv", index=False)
print("Saved. Open results/manual_review_sample.csv and score each output 1-5 for "
      "(a) correctness, (b) fluency, (c) Tanglish style.")
